# 🚀 asyncio — Core Principles & Practical Examples

> **Python's built-in library for writing concurrent code using the async/await syntax.**

---

## 📚 Table of Contents

1. [What is asyncio and why does it exist?](#1-what-is-asyncio)
2. [The Event Loop](#2-the-event-loop)
3. [Coroutines — `async def` and `await`](#3-coroutines)
4. [Tasks — Running coroutines concurrently](#4-tasks)
5. [`asyncio.gather` — Fan-out concurrency](#5-gather)
6. [`asyncio.wait` — Fine-grained control](#6-wait)
7. [Timeouts with `asyncio.timeout`](#7-timeouts)
8. [Queues — Producer / Consumer pattern](#8-queues)
9. [Semaphores — Rate-limiting concurrency](#9-semaphores)
10. [Async Context Managers & Iterators](#10-async-context-managers)
11. [Real-world pattern: Async HTTP with `aiohttp`](#11-real-world-aiohttp)
12. [asyncio in Jupyter — Running in an already-running loop](#12-asyncio-in-jupyter)
13. [Common Pitfalls & Best Practices](#13-pitfalls)
14. [asyncio vs Threading vs Multiprocessing](#14-comparison)
15. [Migrating from `ThreadPoolExecutor` → asyncio](#15-threadpool)
16. [Migrating from `concurrent.futures` → asyncio](#16-concurrent-futures)
17. [Migration Cheat Sheet](#17-cheat-sheet)

---
## 1. What is asyncio?

Python is **single-threaded**. Traditionally, when your program waits for I/O (a network response, a file read, a database query), the thread **blocks** — it literally does nothing while waiting.

**asyncio** solves this with **cooperative multitasking**:
- A single thread runs an **event loop**.
- While one coroutine is *awaiting* I/O, the loop switches to another ready coroutine.
- No OS thread switching overhead, no locks needed for most cases.

### Concurrency vs. Parallelism

| Model | Mechanism | Best for |
|---|---|---|
| `asyncio` | Single thread, cooperative | **I/O-bound** tasks (HTTP, DB, file) |
| `threading` | Multiple OS threads | I/O-bound, simpler tasks |
| `multiprocessing` | Multiple OS processes | **CPU-bound** tasks (number crunching) |

> **Rule of thumb**: Use `asyncio` when you have many I/O-bound operations that can overlap (e.g. calling 10 APIs in parallel). Use `multiprocessing` for CPU-heavy work.

---
## 2. The Event Loop

The **event loop** is the heart of asyncio. It:
- Schedules and runs coroutines.
- Handles I/O callbacks.
- Runs until all scheduled work is complete.

```
┌─────────────────────────────────────────────┐
│              Event Loop                     │
│                                             │
│  ┌──────────┐   ┌──────────┐   ┌─────────┐ │
│  │ Coroutine│   │ Coroutine│   │Callback │ │
│  │    A     │   │    B     │   │  (I/O)  │ │
│  └────┬─────┘   └────┬─────┘   └────┬────┘ │
│       │ await        │ await        │       │
│       └──────────────┴──────────────┘       │
│              (switches between them)        │
└─────────────────────────────────────────────┘
```

In modern Python (3.7+) you rarely interact with the loop directly. Use `asyncio.run()` as the entry point.

In [1]:
import asyncio

# asyncio.run() creates a new event loop, runs the coroutine, then closes the loop.
# This is the standard entry point for asyncio programs.
# NOTE: In Jupyter we use 'await' directly — see section 12.

async def hello_world():
    print("Hello from the event loop!")

# In a standard .py script you would write:
#   asyncio.run(hello_world())
# In Jupyter, the loop is already running, so we just await:
await hello_world()

Hello from the event loop!


---
## 3. Coroutines — `async def` and `await`

A **coroutine** is a special function defined with `async def`. Calling it returns a *coroutine object* (it does **not** run the body yet). The body only runs when it is **awaited**.

```python
async def my_coroutine():
    ...  # body

obj = my_coroutine()   # coroutine object created — body NOT run
await obj              # NOW the body runs
```

The `await` keyword **suspends** the current coroutine, hands control back to the event loop, and resumes when the awaited object is done.

In [2]:
import asyncio
import time

# --- Helper: simulate a slow I/O operation ---
async def fetch_data(name: str, delay: float) -> str:
    """Simulates fetching data that takes `delay` seconds."""
    print(f"  [{name}] Starting fetch (will take {delay}s)...")
    await asyncio.sleep(delay)   # Non-blocking sleep — yields control to the loop
    result = f"{name}: data ready after {delay}s"
    print(f"  [{name}] Done!")
    return result

# --- Sequential execution (NOT the power of asyncio) ---
async def sequential():
    start = time.perf_counter()
    print("=== Sequential ===")
    r1 = await fetch_data("Task-A", 1.0)  # waits 1s
    r2 = await fetch_data("Task-B", 1.0)  # then waits another 1s
    elapsed = time.perf_counter() - start
    print(f"Results: {r1!r}, {r2!r}")
    print(f"Total time: {elapsed:.2f}s  (expected ~2s)\n")

await sequential()

=== Sequential ===
  [Task-A] Starting fetch (will take 1.0s)...
  [Task-A] Done!
  [Task-B] Starting fetch (will take 1.0s)...
  [Task-B] Done!
Results: 'Task-A: data ready after 1.0s', 'Task-B: data ready after 1.0s'
Total time: 2.00s  (expected ~2s)



---
## 4. Tasks — Running Coroutines Concurrently

A **Task** wraps a coroutine and schedules it to run on the event loop *immediately*. Unlike plain `await`, creating a Task lets the event loop pick it up while *you* continue executing.

```python
task = asyncio.create_task(my_coroutine())  # schedules it
# ... do other async work ...
result = await task                          # wait for it and get the result
```

**Key insight**: `asyncio.create_task()` is what gives you *concurrency*. Multiple tasks run interleaved on the single-threaded event loop.

In [3]:
import asyncio
import time

async def concurrent_with_tasks():
    start = time.perf_counter()
    print("=== Concurrent with Tasks ===")

    # Create both tasks immediately — the event loop schedules them concurrently
    task_a = asyncio.create_task(fetch_data("Task-A", 1.0))
    task_b = asyncio.create_task(fetch_data("Task-B", 1.0))

    # Now await them — they ran *concurrently*, so total ≈ 1s, not 2s
    r1 = await task_a
    r2 = await task_b

    elapsed = time.perf_counter() - start
    print(f"Results: {r1!r}, {r2!r}")
    print(f"Total time: {elapsed:.2f}s  (expected ~1s)\n")

await concurrent_with_tasks()

=== Concurrent with Tasks ===
  [Task-A] Starting fetch (will take 1.0s)...
  [Task-B] Starting fetch (will take 1.0s)...
  [Task-A] Done!
  [Task-B] Done!
Results: 'Task-A: data ready after 1.0s', 'Task-B: data ready after 1.0s'
Total time: 1.01s  (expected ~1s)



---
## 5. `asyncio.gather` — Fan-out Concurrency

`asyncio.gather(*awaitables)` is the most common way to run multiple coroutines **concurrently** and collect all results once they're all done.

- Accepts coroutines **or** tasks.
- Returns a list of results in the **same order** as the inputs.
- By default, if one raises an exception the others are **cancelled** (use `return_exceptions=True` to suppress this).

In [4]:
import asyncio
import time

async def demo_gather():
    start = time.perf_counter()
    print("=== asyncio.gather ===")

    # All three run concurrently; total time ≈ max(0.5, 1.0, 1.5) = 1.5s
    results = await asyncio.gather(
        fetch_data("Alpha", 0.5),
        fetch_data("Beta",  1.0),
        fetch_data("Gamma", 1.5),
    )

    elapsed = time.perf_counter() - start
    print(f"All results (in input order): {results}")
    print(f"Total time: {elapsed:.2f}s  (expected ~1.5s)\n")

await demo_gather()

=== asyncio.gather ===
  [Alpha] Starting fetch (will take 0.5s)...
  [Beta] Starting fetch (will take 1.0s)...
  [Gamma] Starting fetch (will take 1.5s)...
  [Alpha] Done!
  [Beta] Done!
  [Gamma] Done!
All results (in input order): ['Alpha: data ready after 0.5s', 'Beta: data ready after 1.0s', 'Gamma: data ready after 1.5s']
Total time: 1.51s  (expected ~1.5s)



In [5]:
# --- return_exceptions=True: handle errors gracefully ---

async def might_fail(name: str, should_fail: bool):
    await asyncio.sleep(0.1)
    if should_fail:
        raise ValueError(f"{name} intentionally failed!")
    return f"{name} succeeded"

async def gather_with_errors():
    results = await asyncio.gather(
        might_fail("job-1", False),
        might_fail("job-2", True),   # this one fails
        might_fail("job-3", False),
        return_exceptions=True,       # exceptions become values, not raises
    )

    for i, res in enumerate(results, 1):
        if isinstance(res, Exception):
            print(f"  job-{i} ERROR: {res}")
        else:
            print(f"  job-{i} OK: {res}")

print("=== gather with return_exceptions ===")
await gather_with_errors()

=== gather with return_exceptions ===
  job-1 OK: job-1 succeeded
  job-2 ERROR: job-2 intentionally failed!
  job-3 OK: job-3 succeeded


---
## 6. `asyncio.wait` — Fine-grained Control

`asyncio.wait(tasks, ...)` gives more control than `gather`:
- Returns two sets: `done` and `pending`.
- Supports `return_when` options:
  - `ALL_COMPLETED` (default) — wait for everything
  - `FIRST_COMPLETED` — return as soon as one finishes
  - `FIRST_EXCEPTION` — return as soon as one raises

Useful for **"first result wins"** patterns or when you want to process results as they arrive.

In [ ]:
import asyncio
import time

async def demo_wait_first_completed():
    start = time.perf_counter()
    print("=== asyncio.wait — FIRST_COMPLETED ===")

    # Create tasks explicitly — asyncio.wait requires tasks, not bare coroutines
    tasks = [
        asyncio.create_task(fetch_data("slow",  2.0)),
        asyncio.create_task(fetch_data("fast",  0.3)),
        asyncio.create_task(fetch_data("medium",1.0)),
    ]

    done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)

    elapsed = time.perf_counter() - start
    print(f"\nFirst to finish (after {elapsed:.2f}s):")
    for t in done:
        print(f"  result = {t.result()!r}")

    # Cancel remaining tasks to avoid "Task was destroyed but it is pending" warnings
    print(f"Cancelling {len(pending)} remaining tasks...")
    for t in pending:
        t.cancel()
    await asyncio.gather(*pending, return_exceptions=True)   # let cancellations propagate

await demo_wait_first_completed()

---
## 7. Timeouts with `asyncio.timeout`

Two main approaches to enforce a maximum wait time:

| API | Python version | Notes |
|---|---|---|
| `asyncio.timeout(seconds)` | 3.11+ | Context manager, clean cancellation |
| `asyncio.wait_for(coro, timeout=seconds)` | All | Wraps a single awaitable |

On timeout:
- `wait_for` raises `asyncio.TimeoutError`.
- `asyncio.timeout` raises `TimeoutError` (built-in, same thing in 3.11+).

In [6]:
import asyncio
import sys

async def slow_operation():
    await asyncio.sleep(5)  # Takes 5 seconds
    return "finished"

# --- wait_for (works on all modern Python versions) ---
async def demo_timeout_wait_for():
    print("=== asyncio.wait_for with timeout ===")
    try:
        result = await asyncio.wait_for(slow_operation(), timeout=1.0)
        print(f"Result: {result}")
    except asyncio.TimeoutError:
        print("  ⏰ Timed out after 1 second — operation cancelled!")

await demo_timeout_wait_for()

# --- asyncio.timeout context manager (Python 3.11+) ---
async def demo_timeout_context():
    print("\n=== asyncio.timeout context manager (3.11+) ===")
    if sys.version_info < (3, 11):
        print("  Skipped — requires Python 3.11+")
        return

    try:
        async with asyncio.timeout(1.0):
            await slow_operation()
    except TimeoutError:
        print("  ⏰ Context-manager timeout triggered!")

await demo_timeout_context()

=== asyncio.wait_for with timeout ===
  ⏰ Timed out after 1 second — operation cancelled!

=== asyncio.timeout context manager (3.11+) ===
  ⏰ Context-manager timeout triggered!


---
## 8. Queues — Producer / Consumer Pattern

`asyncio.Queue` is a thread-safe FIFO queue designed for coordinating producers and consumers inside the event loop.

**Typical use case**: Many API calls come in (producers put work on the queue); a fixed number of workers drain it (consumers get items and process them).

Key methods:
- `await queue.put(item)` — add item (blocks if queue is full)
- `await queue.get()` — remove and return an item (blocks until available)
- `queue.task_done()` — signal that a gotten item has been processed
- `await queue.join()` — block until all items have been `task_done()`'d

In [7]:
import asyncio
import random

async def producer(queue: asyncio.Queue, n_items: int):
    """Puts `n_items` work items onto the queue."""
    for i in range(n_items):
        item = f"task-{i+1}"
        await queue.put(item)
        print(f"  📥 Produced: {item}")
        await asyncio.sleep(0.1)   # simulate arrival rate
    print("  Producer finished.")

async def consumer(worker_id: int, queue: asyncio.Queue):
    """Continuously consumes items from the queue."""
    while True:
        item = await queue.get()       # blocks until an item is available
        process_time = random.uniform(0.1, 0.4)
        await asyncio.sleep(process_time)   # simulate work
        print(f"  ✅ Worker-{worker_id} processed {item!r} in {process_time:.2f}s")
        queue.task_done()              # signal this item is done

async def demo_queue():
    print("=== Producer / Consumer with asyncio.Queue ===")
    queue = asyncio.Queue(maxsize=5)   # buffer at most 5 items

    # Spin up 3 consumers in the background
    workers = [
        asyncio.create_task(consumer(wid, queue))
        for wid in range(1, 4)
    ]

    # Run the producer
    await producer(queue, n_items=8)

    # Wait until all produced items are fully processed
    await queue.join()
    print("  All items processed!")

    # Cancel the idle worker tasks
    for w in workers:
        w.cancel()
    await asyncio.gather(*workers, return_exceptions=True)

await demo_queue()

=== Producer / Consumer with asyncio.Queue ===
  📥 Produced: task-1
  📥 Produced: task-2
  ✅ Worker-1 processed 'task-1' in 0.20s
  📥 Produced: task-3
  📥 Produced: task-4
  ✅ Worker-3 processed 'task-3' in 0.18s
  📥 Produced: task-5
  ✅ Worker-2 processed 'task-2' in 0.32s
  📥 Produced: task-6
  ✅ Worker-2 processed 'task-5' in 0.11s
  ✅ Worker-1 processed 'task-4' in 0.29s
  📥 Produced: task-7
  ✅ Worker-2 processed 'task-6' in 0.15s
  📥 Produced: task-8
  ✅ Worker-3 processed 'task-7' in 0.17s
  Producer finished.
  ✅ Worker-1 processed 'task-8' in 0.32s
  All items processed!


---
## 9. Semaphores — Rate-limiting Concurrency

An `asyncio.Semaphore` limits the number of coroutines that can be in a critical section simultaneously.

**Classic use-case**: You want to make 50 API calls but the server only allows 5 simultaneous connections.

In [ ]:
import asyncio
import time

# Simulate an API call that takes ~0.5s
async def api_call(sem: asyncio.Semaphore, call_id: int) -> str:
    async with sem:   # only MAX_CONCURRENT tasks can be inside this block at once
        print(f"  🔑 [{call_id:02d}] acquired semaphore — calling API...")
        await asyncio.sleep(0.5)
        print(f"  ✔  [{call_id:02d}] done")
        return f"response-{call_id}"

async def demo_semaphore():
    MAX_CONCURRENT = 3   # allow at most 3 simultaneous API calls
    N_CALLS = 9

    print(f"=== Semaphore: {N_CALLS} calls, max {MAX_CONCURRENT} at a time ===")
    sem = asyncio.Semaphore(MAX_CONCURRENT)

    start = time.perf_counter()
    results = await asyncio.gather(
        *[api_call(sem, i) for i in range(1, N_CALLS + 1)]
    )
    elapsed = time.perf_counter() - start

    # 9 calls / 3 concurrent = 3 batches × 0.5s ≈ 1.5s
    print(f"\nTotal time: {elapsed:.2f}s  (expected ~{0.5 * (N_CALLS // MAX_CONCURRENT)}s)")

await demo_semaphore()

---
## 10. Async Context Managers & Iterators

### Async Context Managers (`async with`)

Any class implementing `__aenter__` and `__aexit__` can be used as an async context manager. This is used by database connection pools, HTTP sessions, etc.

### Async Iterators / Generators (`async for`, `async yield`)

An async generator uses `async def` + `yield` to produce values asynchronously, perfect for streaming results.

In [ ]:
import asyncio

# ── Async Context Manager ──────────────────────────────────────────────────────
class AsyncDBConnection:
    """Simulates an async database connection with setup and teardown."""

    async def __aenter__(self):
        print("  📂 Opening DB connection...")
        await asyncio.sleep(0.1)   # simulated network connect
        return self                # returned as the 'as' variable

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print("  📁 Closing DB connection...")
        await asyncio.sleep(0.05)  # simulated close
        return False               # don't suppress exceptions

    async def query(self, sql: str) -> list:
        await asyncio.sleep(0.1)
        return [{"row": 1, "sql": sql}, {"row": 2, "sql": sql}]

async def demo_async_context_manager():
    print("=== Async Context Manager ===")
    async with AsyncDBConnection() as db:
        rows = await db.query("SELECT * FROM users")
        print(f"  Got {len(rows)} rows")
    print("  Outside the context — connection is closed.")

await demo_async_context_manager()

In [ ]:
import asyncio

# ── Async Generator ────────────────────────────────────────────────────────────
async def stream_events(source: str, count: int):
    """Yields events one at a time, simulating a streaming API or WebSocket."""
    for i in range(count):
        await asyncio.sleep(0.2)   # wait for next event from the server
        yield {"source": source, "event_id": i, "data": f"payload-{i}"}

async def demo_async_generator():
    print("=== Async Generator / Iterator ===")
    # 'async for' iterates over values as they become available
    async for event in stream_events("sensor-A", 5):
        print(f"  📡 Received: {event}")

await demo_async_generator()

---
## 11. Real-world Pattern: Async HTTP with `aiohttp`

`aiohttp` is the de-facto async HTTP client library. It uses async context managers for sessions and responses.

In [ ]:
# First, ensure aiohttp is available:
# !pip install aiohttp  (or: uv add aiohttp)

try:
    import aiohttp
    AIOHTTP_AVAILABLE = True
except ImportError:
    AIOHTTP_AVAILABLE = False
    print("ℹ️  aiohttp not installed. Run: uv add aiohttp")

import asyncio
import time

URLS = [
    "https://httpbin.org/delay/1",  # each takes ~1s
    "https://httpbin.org/delay/1",
    "https://httpbin.org/delay/1",
]

async def fetch_url(session: "aiohttp.ClientSession", url: str) -> dict:
    """Fetch a single URL and return parsed JSON."""
    print(f"  → GET {url}")
    async with session.get(url) as response:
        data = await response.json()
        print(f"  ← {response.status} {url}")
        return data

async def fetch_all_urls():
    if not AIOHTTP_AVAILABLE:
        print("Skipped — aiohttp not available.")
        return

    start = time.perf_counter()
    print("=== Concurrent HTTP with aiohttp ===")

    # One shared session is efficient (connection pooling, cookie handling)
    async with aiohttp.ClientSession() as session:
        results = await asyncio.gather(
            *[fetch_url(session, url) for url in URLS]
        )

    elapsed = time.perf_counter() - start
    print(f"\nFetched {len(results)} URLs in {elapsed:.2f}s  (expected ~1s, not {len(URLS)}s)")

await fetch_all_urls()

---
## 12. asyncio in Jupyter — Running in an Already-Running Loop

Jupyter/IPython already runs an event loop internally. This means you **cannot** call `asyncio.run()` from a notebook cell — it will raise a `RuntimeError: This event loop is already running`.

The solution: **just `await` at the top level** of the cell. Jupyter supports this natively.

```python
# ✅ In Jupyter — just await directly:
result = await my_coroutine()

# ❌ Do NOT do this in Jupyter:
result = asyncio.run(my_coroutine())   # RuntimeError!
```

In a regular `.py` script, `asyncio.run()` is the correct entry point.

In [ ]:
import asyncio

# Detect whether we're inside an already-running event loop
try:
    loop = asyncio.get_running_loop()
    print(f"✅ Running inside an existing event loop: {loop}")
    print("   → Use 'await' directly in Jupyter cells.")
except RuntimeError:
    loop = None
    print("No running loop — use asyncio.run() as entry point (standard .py script).")

# This top-level await works perfectly in Jupyter:
result = await asyncio.sleep(0, result="Hello from top-level await!")
print(result)

---
## 13. Common Pitfalls & Best Practices

### ❌ Pitfall 1: Forgetting `await`

```python
# Wrong — creates the coroutine object but never runs it!
result = my_coroutine()

# Correct
result = await my_coroutine()
```

### ❌ Pitfall 2: Blocking calls in async code

```python
# Wrong — time.sleep() BLOCKS the entire event loop!
async def bad():
    time.sleep(2)   # blocks ALL other coroutines for 2s

# Correct — yields control while waiting
async def good():
    await asyncio.sleep(2)
```

For truly CPU-bound work, use `loop.run_in_executor()` to offload to a thread pool.

### ❌ Pitfall 3: Not awaiting `Task` cancellations

```python
# Wrong — cancellation is a request, not instant
task.cancel()
# The task may still be alive here!

# Correct — await after cancel to actually wait for termination
task.cancel()
await task   # (or await asyncio.gather(task, return_exceptions=True))
```

### ✅ Best Practices Summary

| Do | Don't |
|---|---|
| Use `asyncio.run()` as the single entry point (in scripts) | Call `asyncio.run()` in Jupyter |
| Use `asyncio.gather()` for fan-out concurrency | Sequential `await` in a loop when tasks are independent |
| Use `Semaphore` to limit concurrency | Launch unlimited tasks that overwhelm external systems |
| Use `asyncio.sleep()` in async code | Use `time.sleep()` in async code |
| Await task cancellations | Forget to clean up pending tasks |
| Use one shared `aiohttp.ClientSession` per session | Create a new session per request |

In [ ]:
# ── Blocking CPU work offloaded to a thread pool ──────────────────────────────
import asyncio
import time

def heavy_cpu_task(n: int) -> int:
    """A pure CPU-bound function (no async)."""
    total = 0
    for i in range(n):
        total += i * i
    return total

async def demo_run_in_executor():
    print("=== run_in_executor for CPU-bound work ===")
    loop = asyncio.get_running_loop()

    # Offload blocking work to a thread — doesn't block the event loop
    result = await loop.run_in_executor(
        None,                 # None = default ThreadPoolExecutor
        heavy_cpu_task,       # function to call
        1_000_000             # argument
    )
    print(f"  Sum of squares 0..999999 = {result}")
    print("  Other coroutines could have run while this was computing!")

await demo_run_in_executor()

---
## 🎯 Summary

```
Concept              | API                         | Use it when...
─────────────────────┼─────────────────────────────┼──────────────────────────────────────
Coroutine            | async def / await           | Any async function definition
Entry point (script) | asyncio.run()               | Top-level in a .py file
Concurrent tasks     | asyncio.create_task()       | Fire-and-forget, fine control
Fan-out              | asyncio.gather()            | Run N tasks, collect all results
First winner         | asyncio.wait(FIRST_COMP..)  | Race multiple awaitables
Timeout              | asyncio.wait_for()          | Enforce a max wait time
Work queue           | asyncio.Queue               | Producer/consumer pipelines
Rate limiting        | asyncio.Semaphore           | Max N concurrent connections
Streaming            | async def + yield           | Process results as they arrive
CPU work             | loop.run_in_executor()      | Don't block the event loop
```

> **The golden rule of asyncio**: A coroutine that blocks (CPU work, `time.sleep`, synchronous I/O) blocks the **entire event loop** and defeats the purpose of async. Always use the async equivalent or offload to `run_in_executor`.

---
## 14. asyncio vs Threading vs Multiprocessing

Python offers three concurrency models. The right choice depends entirely on **what is bottlenecking your program**.

### The GIL — why three models exist

The **Global Interpreter Lock (GIL)** is a mutex inside CPython that allows only **one thread to run Python bytecode at a time**.

- **threading** — Multiple OS threads, but the GIL limits true CPU parallelism. Works well for I/O because threads release the GIL while waiting on system calls.
- **multiprocessing** — Separate processes, each with its own GIL. True CPU parallelism, but higher memory and startup overhead.
- **asyncio** — Single thread, cooperative multitasking. Zero thread-switching overhead; yields control at every `await`. Cannot parallelize CPU work.

### Decision flowchart

```
What bottlenecks your program?
     │
     ├── I/O bound (HTTP, DB, file, socket)
     │       ├── asyncio   ✅ best – lowest overhead, highest throughput
     │       └── threading ✅ ok  – simpler to retrofit existing sync code
     │
     └── CPU bound (ML, math, image processing)
             ├── multiprocessing           ✅ best – bypasses GIL
             ├── asyncio + ProcessPool     ✅ async-friendly wrapper
             └── threading                 ❌ GIL prevents true parallelism
```

### Feature matrix

| Feature | `asyncio` | `threading` | `multiprocessing` |
|---|---|---|---|
| True CPU parallelism | ❌ Single thread | ❌ GIL-limited | ✅ Separate processes |
| I/O concurrency | ✅ Excellent | ✅ Good | ⚠️ Overkill |
| Memory per unit | ✅ Negligible | ⚠️ ~8 MB/thread | ❌ Full process copy |
| Startup cost | ✅ None | ⚠️ Small | ❌ Significant |
| Shared mutable state | ✅ Easy (1 thread) | ⚠️ Need locks | ❌ Need IPC/Manager |
| Debugging | ✅ Predictable order | ❌ Race conditions | ❌ Complex IPC |
| Error propagation | ✅ Structured | ⚠️ Manual | ⚠️ Manual |

In [ ]:
# Live benchmark: I/O-bound task – 20 items, each sleeps 0.2 s
# Expected: sequential ~4 s, concurrent models ~0.2 s

import asyncio, time, threading
from concurrent.futures import ThreadPoolExecutor

ITEMS = list(range(20))
DELAY = 0.2

def sync_work(item_id: int) -> str:
    time.sleep(DELAY)
    return f'done-{item_id}'

async def async_work(item_id: int) -> str:
    await asyncio.sleep(DELAY)   # non-blocking
    return f'done-{item_id}'

# 1. Sequential
def run_sequential():
    return [sync_work(i) for i in ITEMS]

# 2. threading.Thread (manual)
def run_threading():
    results = [None] * len(ITEMS)
    def worker(idx):
        results[idx] = sync_work(idx)
    threads = [threading.Thread(target=worker, args=(i,)) for i in ITEMS]
    for t in threads: t.start()
    for t in threads: t.join()
    return results

# 3. ThreadPoolExecutor
def run_thread_pool():
    with ThreadPoolExecutor(max_workers=20) as pool:
        return list(pool.map(sync_work, ITEMS))

# 4. asyncio + run_in_executor (keeps existing sync code)
async def run_asyncio_executor():
    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=20) as pool:
        return await asyncio.gather(
            *[loop.run_in_executor(pool, sync_work, i) for i in ITEMS]
        )

# 5. Pure asyncio (native non-blocking I/O)
async def run_pure_asyncio():
    return await asyncio.gather(*[async_work(i) for i in ITEMS])

# Run & report
print(f"{'Model':<35} {'Time (s)':>10} {'Speedup':>10}")
print('-' * 58)

baseline = None
for label, fn, is_coro in [
    ('1. Sequential',         run_sequential,       False),
    ('2. threading.Thread',   run_threading,        False),
    ('3. ThreadPoolExecutor', run_thread_pool,      False),
    ('4. asyncio + executor', run_asyncio_executor, True),
    ('5. Pure asyncio',       run_pure_asyncio,     True),
]:
    t0 = time.perf_counter()
    result = await fn() if is_coro else fn()
    elapsed = time.perf_counter() - t0
    if baseline is None:
        baseline = elapsed
        speedup = '—'
    else:
        speedup = f'{baseline / elapsed:.1f}×'
    print(f'{label:<35} {elapsed:>10.3f} {speedup:>10}')

---
## 15. Migrating from `ThreadPoolExecutor` → asyncio

The migration path has two flavours:

| Approach | When to use |
|---|---|
| **Bridge** (`run_in_executor`) | Can't rewrite the sync function (third-party lib, legacy code) |
| **Full rewrite** (`async def`) | Owning the code and an async library exists (e.g. `aiohttp` instead of `requests`) |

### Bridge pattern (keep sync code)

```python
# Before — ThreadPoolExecutor
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=5) as pool:
    futures = [pool.submit(requests.get, url) for url in urls]
    results = [f.result() for f in futures]        # blocks

# After — asyncio bridge (sync fn unchanged)
import asyncio
from concurrent.futures import ThreadPoolExecutor

async def main():
    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=5) as pool:
        results = await asyncio.gather(
            *[loop.run_in_executor(pool, requests.get, url) for url in urls]
        )
```

### Full rewrite (native async)

```python
# Before
def fetch(url):
    return requests.get(url).text          # sync, blocking

# After
async def fetch(session, url):
    async with session.get(url) as r:      # async, non-blocking
        return await r.text()

async def main():
    async with aiohttp.ClientSession() as session:
        results = await asyncio.gather(
            *[fetch(session, url) for url in urls]
        )
```

In [ ]:
import asyncio, time
from concurrent.futures import ThreadPoolExecutor

# ----- existing sync function (pretend we can't change it) -----
def legacy_fetch(item_id: int) -> str:
    time.sleep(0.3)   # blocking – simulates requests.get()
    return f'item-{item_id}: fetched'

# ----- bridge: call sync function from async code ---------------
async def async_legacy_fetch(pool, item_id: int) -> str:
    loop = asyncio.get_running_loop()
    # Dispatched to the thread pool; event loop is NOT blocked
    return await loop.run_in_executor(pool, legacy_fetch, item_id)

# ----- compare ------------------------------------------------
ITEMS = list(range(1, 6))

# Old way: ThreadPoolExecutor (synchronous context)
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as pool:
    thread_results = list(pool.map(legacy_fetch, ITEMS))
t_thread = time.perf_counter() - t0

# New way: asyncio + executor bridge
t1 = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as pool:
    async_results = await asyncio.gather(
        *[async_legacy_fetch(pool, i) for i in ITEMS]
    )
t_async = time.perf_counter() - t1

print(f'ThreadPoolExecutor (sync)   → {t_thread:.2f}s  {thread_results}')
print(f'asyncio + executor (async)  → {t_async:.2f}s  {async_results}')
print('Same speed, but the async version composes with other coroutines!')

---
## 16. Migrating from `concurrent.futures` → asyncio

### API equivalence table

| `concurrent.futures` | asyncio equivalent | Notes |
|---|---|---|
| `executor.submit(fn, arg)` | `loop.run_in_executor(pool, fn, arg)` | Returns an awaitable |
| `executor.map(fn, items)` | `asyncio.gather(*[...])` | Collect all results |
| `futures.as_completed(fs)` | `asyncio.as_completed(coros)` | Yield in completion order |
| `futures.wait(fs)` | `asyncio.wait(tasks)` | done / pending sets |
| `f.result(timeout=N)` | `await asyncio.wait_for(coro, N)` | Raises `TimeoutError` |
| `f.cancel()` | `task.cancel()` | Cancellation is cooperative |
| `ThreadPoolExecutor(n)` | `asyncio.Semaphore(n)` + gather | Limit concurrency |
| `ProcessPoolExecutor(n)` | `run_in_executor(ProcessPoolExecutor(n), fn)` | CPU-bound work |

### `as_completed` – process results as they arrive

In [ ]:
import asyncio, time
from concurrent.futures import ThreadPoolExecutor, as_completed

def work(n: int) -> int:
    time.sleep(n * 0.1)   # variable latency
    return n * n

ITEMS = [5, 1, 3, 2, 4]   # intentionally out of order

# ── concurrent.futures: as_completed ────────────────────────────
print('=== concurrent.futures.as_completed ===')
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as pool:
    fs = {pool.submit(work, n): n for n in ITEMS}
    for f in as_completed(fs):
        n = fs[f]
        print(f'  finished n={n}  result={f.result()}')
print(f'  time: {time.perf_counter() - t0:.2f}s\n')

# ── asyncio.as_completed equivalent ─────────────────────────────
async def async_work(n: int) -> int:
    await asyncio.sleep(n * 0.1)
    return n * n

print('=== asyncio.as_completed ===')
t1 = time.perf_counter()
# asyncio.as_completed yields awaitables in the order they FINISH
for coro in asyncio.as_completed([async_work(n) for n in ITEMS]):
    result = await coro
    print(f'  finished  result={result}')
print(f'  time: {time.perf_counter() - t1:.2f}s')

In [ ]:
# ProcessPoolExecutor – CPU-bound work from async code
# (Must be top-level importable on Windows; define the fn in a real module
#  or use the if __name__ == '__main__' guard in scripts.)

import asyncio, math, time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

def cpu_work(_n: int) -> float:
    """CPU-intensive: sum of square roots to 5 × 10^5."""
    return sum(math.sqrt(i) for i in range(500_000))

CPU_ITEMS = list(range(4))

# Sequential (baseline)
t0 = time.perf_counter()
seq = [cpu_work(i) for i in CPU_ITEMS]
t_seq = time.perf_counter() - t0

# ThreadPoolExecutor – GIL limits true CPU parallelism
t1 = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    thr = list(pool.map(cpu_work, CPU_ITEMS))
t_thr = time.perf_counter() - t1

# asyncio + ProcessPoolExecutor – CPU parallelism inside async code
async def run_in_process_pool():
    loop = asyncio.get_running_loop()
    with ProcessPoolExecutor(max_workers=4) as pool:
        return await asyncio.gather(
            *[loop.run_in_executor(pool, cpu_work, i) for i in CPU_ITEMS]
        )

t2 = time.perf_counter()
proc = await run_in_process_pool()
t_proc = time.perf_counter() - t2

print(f"{'Model':<40} {'Time':>8}  Notes")
print('-' * 70)
print(f"{'Sequential':<40} {t_seq:>8.3f}s  baseline")
print(f"{'ThreadPoolExecutor (CPU-bound)':<40} {t_thr:>8.3f}s  GIL limits gains")
print(f"{'asyncio + ProcessPoolExecutor':<40} {t_proc:>8.3f}s  true parallelism ✅")

---
## 17. Migration Cheat Sheet

```python
# ═══════════════════════════════════════════════════════════
#  FROM threading / concurrent.futures  →  TO asyncio
# ═══════════════════════════════════════════════════════════

# 1. SINGLE TASK ─────────────────────────────────────────────
# Old:
with ThreadPoolExecutor() as pool:
    result = pool.submit(my_fn, arg).result()   # blocks

# New (sync fn, can't rewrite):
result = await loop.run_in_executor(None, my_fn, arg)

# New (own the code – rewrite as async):
result = await my_async_fn(arg)


# 2. MANY TASKS ──────────────────────────────────────────────
# Old:
with ThreadPoolExecutor(max_workers=10) as pool:
    results = list(pool.map(my_fn, items))

# New:
results = await asyncio.gather(*[my_async_fn(i) for i in items])


# 3. PROCESS RESULTS AS THEY ARRIVE ──────────────────────────
# Old:
for f in concurrent.futures.as_completed(futures):
    print(f.result())

# New:
for coro in asyncio.as_completed(coroutines):
    print(await coro)


# 4. TIMEOUT ─────────────────────────────────────────────────
# Old:
result = pool.submit(my_fn).result(timeout=5)  # raises TimeoutError

# New:
result = await asyncio.wait_for(my_async_fn(), timeout=5)


# 5. LIMIT CONCURRENCY ───────────────────────────────────────
# Old:
with ThreadPoolExecutor(max_workers=5) as pool:   # 5 threads max
    ...

# New:
sem = asyncio.Semaphore(5)
async def limited(item):
    async with sem:
        return await my_async_fn(item)
results = await asyncio.gather(*[limited(i) for i in items])


# 6. CANCEL TASKS ─────────────────────────────────────────────
# Old:
future.cancel()     # best-effort only for futures

# New:  (must await to let CancelledError propagate cleanly)
task.cancel()
await asyncio.gather(task, return_exceptions=True)


# 7. CPU-BOUND WORK ───────────────────────────────────────────
# I/O-bound sync code → thread pool (GIL released during I/O)
await loop.run_in_executor(ThreadPoolExecutor(), blocking_io_fn, arg)

# CPU-bound sync code → process pool (bypasses GIL entirely)
await loop.run_in_executor(ProcessPoolExecutor(), cpu_heavy_fn, arg)
```

> **Rule of thumb**: prefer `asyncio` for new I/O-bound code. Use `run_in_executor`
> to bridge legacy sync code without rewriting it. Reach for `ProcessPoolExecutor`
> only when profiling confirms a CPU bottleneck.